In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
from cartopy.feature import LAND
import matplotlib.dates as mdates
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
from limosat_analysis import open_limosat

DATA_PATH = Path("../data/201905_202007.parquet")  # Or a QC SQLite file

CENTER_X_M = 700_000
CENTER_Y_M = -800_000
BUFFER_M = 200_000
TIME_START = "2020-02-20"
TIME_STOP = "2020-02-25"

PLOT_PAD_M = 75_000
MIN_PLOT_SEGMENT_LENGTH_M = 1_000


## Select trajectories passing through a space-time window

Install this repository with `pip install -e ".[notebooks]"` in the notebook environment. Change the source, centre point, buffer, and UTC dates in the first cell. Time includes `TIME_START` and excludes `TIME_STOP`; box boundaries are inclusive EPSG:3413 metres.

The reader finds trajectories with at least one observation inside the window, then fetches their complete histories within this source period. With displacement Parquet, the observation view contains linked endpoints; singleton trajectories are unavailable. This is an observation-in-box query, not an intersection test for segments crossing the box without an endpoint inside.


In [ ]:
data = open_limosat(DATA_PATH, crs="EPSG:3413")
data.metadata


In [ ]:
def bounds_from_center(center_x_m, center_y_m, buffer_m):
    return (center_x_m - buffer_m, center_x_m + buffer_m), (center_y_m - buffer_m, center_y_m + buffer_m)


## Run the selection


In [ ]:
xlim, ylim = bounds_from_center(CENTER_X_M, CENTER_Y_M, BUFFER_M)
selected_trajectories, domain_points = data.read_trajectories_through(
    start=TIME_START,
    stop=TIME_STOP,
    xlim=xlim,
    ylim=ylim,
    linked_only=True,
)
trajectory_ids = np.sort(domain_points.trajectory_id.unique())

summary = pd.Series({
    "time_start": pd.to_datetime(TIME_START, utc=True),
    "time_stop_exclusive": pd.to_datetime(TIME_STOP, utc=True),
    "center_x_m": CENTER_X_M,
    "center_y_m": CENTER_Y_M,
    "buffer_m": BUFFER_M,
    "x_min_m": xlim[0],
    "x_max_m": xlim[1],
    "y_min_m": ylim[0],
    "y_max_m": ylim[1],
    "domain_points": len(domain_points),
    "selected_trajectories": len(trajectory_ids),
    "selected_trajectory_points": len(selected_trajectories),
})
summary


In [ ]:
trajectory_summary = selected_trajectories.groupby("trajectory_id").agg(
    first_time=("time_utc", "min"),
    last_time=("time_utc", "max"),
    n_points=("time_utc", "size"),
    domain_points=("trajectory_id", lambda values: int(values.index.isin(domain_points.index).sum())),
)
trajectory_summary["duration_days"] = (
    trajectory_summary.last_time - trajectory_summary.first_time
).dt.total_seconds() / 86_400
trajectory_summary.sort_values(["domain_points", "n_points"], ascending=False).head(10)


## Plot the query


In [ ]:
def point_in_query(points, x_col, y_col, time_col):
    return (
        points[time_col].between(pd.to_datetime(TIME_START, utc=True), pd.to_datetime(TIME_STOP, utc=True), inclusive="left")
        & points[x_col].between(*xlim)
        & points[y_col].between(*ylim)
    )


projection = ccrs.NorthPolarStereo(central_longitude=-45, true_scale_latitude=70.0)
plot_segments = data.read_displacements(trajectory_ids=trajectory_ids)
start_in = point_in_query(plot_segments, "x0_m", "y0_m", "start_time_utc")
end_in = point_in_query(plot_segments, "x1_m", "y1_m", "end_time_utc")
plot_segments = plot_segments.loc[(start_in | end_in) & (plot_segments.elapsed_seconds > 0)].copy()
plot_segments["segment_time"] = plot_segments.start_time_utc + (plot_segments.end_time_utc - plot_segments.start_time_utc) / 2
plot_segments["segment_length_m"] = plot_segments.distance_m
plot_segments = plot_segments.loc[plot_segments.segment_length_m >= MIN_PLOT_SEGMENT_LENGTH_M].copy()

time_norm = Normalize(mdates.date2num(pd.to_datetime(TIME_START, utc=True)), mdates.date2num(pd.to_datetime(TIME_STOP, utc=True)))
cmap = "viridis"

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": projection}, constrained_layout=True)
ax.add_feature(LAND, zorder=3, edgecolor="black", facecolor="lightgray", linewidth=0.5)

if not plot_segments.empty:
    xy = plot_segments[["x0_m", "y0_m", "x1_m", "y1_m"]].to_numpy(dtype=float).reshape(-1, 2, 2)
    lines = LineCollection(xy, cmap=cmap, norm=time_norm, linewidths=0.65, alpha=0.9, capstyle="butt", transform=projection, zorder=2)
    lines.set_array(mdates.date2num(plot_segments.segment_time))
    ax.add_collection(lines)
    color_source = lines
else:
    color_source = ScalarMappable(norm=time_norm, cmap=cmap)
    color_source.set_array([])

box_x = [xlim[0], xlim[1], xlim[1], xlim[0], xlim[0]]
box_y = [ylim[0], ylim[0], ylim[1], ylim[1], ylim[0]]
ax.plot(box_x, box_y, color="black", linewidth=1.2, transform=projection, zorder=4, label="query box")

ax.set_extent(
    [xlim[0] - PLOT_PAD_M, xlim[1] + PLOT_PAD_M, ylim[0] - PLOT_PAD_M, ylim[1] + PLOT_PAD_M],
    crs=projection,
)
ax.set_xlabel("x EPSG:3413 (m)")
ax.set_ylabel("y EPSG:3413 (m)")
ax.set_title(
    f"{pd.to_datetime(TIME_START, utc=True):%Y-%m-%d} to {pd.to_datetime(TIME_STOP, utc=True):%Y-%m-%d}"
)
ax.legend(frameon=False, loc="upper right")

cbar = fig.colorbar(color_source, ax=ax, shrink=0.8, label="date")
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

plt.show()
